Test BasicMotions → HDF5 (BioFoundation)

This script validates that the generated train/val/test HDF5 files:
- contain at least one top-level group
- each group contains datasets 'X' and 'y'
- X has shape (N, C, T) and float dtype
- y is 1D integer labels in 0..K-1 (contiguous)
- (C, T) are consistent across splits
- no NaN/Inf

How to run:
  python test_basicmotions_hdf5.py --base_dir "${DATA_PATH}/UEA_MTS/BasicMotions/processed"

If you don't pass --base_dir, it will try to use $DATA_PATH automatically.

In [ ]:
import os
import json
import argparse
from collections import Counter

import numpy as np
import h5py


def inspect_h5(path: str) -> dict:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"File not found: {path}")

    info = {}
    with h5py.File(path, "r") as f:
        groups = list(f.keys())
        if len(groups) < 1:
            raise RuntimeError(f"No top-level groups found in {path}")

        info["groups"] = groups
        per_group = {}

        for g in groups:
            grp = f[g]
            if "X" not in grp:
                raise RuntimeError(f"Group '{g}' missing dataset 'X' in {path}")
            if "y" not in grp:
                raise RuntimeError(f"Group '{g}' missing dataset 'y' in {path}")

            X = grp["X"]
            y = grp["y"]

            per_group[g] = {
                "X_shape": tuple(X.shape),
                "X_dtype": str(X.dtype),
                "y_shape": tuple(y.shape),
                "y_dtype": str(y.dtype),
                "X_min": float(np.min(X[:])) if X.size else None,
                "X_max": float(np.max(X[:])) if X.size else None,
            }

        info["per_group"] = per_group

    return info


def load_group(path: str, group: str | None = None):
    with h5py.File(path, "r") as f:
        groups = list(f.keys())
        if not groups:
            raise RuntimeError(f"No groups in {path}")
        if group is None:
            group = groups[0]
        grp = f[group]
        X = grp["X"][:]
        y = grp["y"][:]
    return X, y, group


def assert_3d(X: np.ndarray, name: str) -> None:
    if X.ndim != 3:
        raise AssertionError(f"{name}: expected 3D array (N,C,T), got shape {X.shape}")


def label_report(y: np.ndarray, name: str) -> np.ndarray:
    y = np.asarray(y)
    if y.ndim != 1:
        raise AssertionError(f"{name}: expected 1D labels, got {y.shape}")
    if not np.issubdtype(y.dtype, np.integer):
        raise AssertionError(f"{name}: expected integer labels, got {y.dtype}")

    classes = np.unique(y)
    print(f"{name}: N={len(y)}  classes={classes.tolist()}  K={len(classes)}")
    print("  min/max:", int(classes.min()), int(classes.max()))
    counts = Counter(y.tolist())
    for k in sorted(counts):
        print(f"  class {k}: {counts[k]}")
    return classes


def numeric_sanity(X: np.ndarray, name: str) -> None:
    X = np.asarray(X)
    if not np.isfinite(X).all():
        raise AssertionError(f"{name}: contains NaN/Inf")
    print(
        f"{name}: mean={X.mean():.4f}, std={X.std():.4f}, min={X.min():.4f}, max={X.max():.4f}"
    )


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--base_dir",
        type=str,
        default=None,
        help="Directory containing train.h5/val.h5/test.h5 (defaults to $DATA_PATH/UEA_MTS/BasicMotions/processed)",
    )
    parser.add_argument(
        "--group",
        type=str,
        default=None,
        help="Optional: choose a specific HDF5 group name (defaults to first group in each file).",
    )
    args = parser.parse_args()

    data_path = os.environ.get("DATA_PATH")
    if args.base_dir is None:
        if data_path is None:
            raise RuntimeError(
                "No --base_dir provided and DATA_PATH is not set. "
                "Set DATA_PATH or pass --base_dir explicitly."
            )
        base_dir = os.path.join(data_path, "UEA_MTS", "BasicMotions", "processed")
    else:
        base_dir = args.base_dir

    train_h5 = os.path.join(base_dir, "train.h5")
    val_h5 = os.path.join(base_dir, "val.h5")
    test_h5 = os.path.join(base_dir, "test.h5")
    meta_json = os.path.join(base_dir, "meta.json")

    print("Base dir:", base_dir)
    print("Train:", train_h5)
    print("Val  :", val_h5)
    print("Test :", test_h5)
    print()

    # --- Inspect structure ---
    print("=== HDF5 STRUCTURE ===")
    for p in [train_h5, val_h5, test_h5]:
        info = inspect_h5(p)
        print(os.path.basename(p), "groups:", info["groups"])
        # Print first group's details
        g0 = info["groups"][0]
        print(" ", g0, info["per_group"][g0])
    print()

    # --- Load first group and run checks ---
    Xtr, ytr, gtr = load_group(train_h5, args.group)
    Xva, yva, gva = load_group(val_h5, args.group)
    Xte, yte, gte = load_group(test_h5, args.group)

    print("=== LOADED ARRAYS ===")
    print("Train group:", gtr, "X", Xtr.shape, Xtr.dtype, "y", ytr.shape, ytr.dtype)
    print("Val   group:", gva, "X", Xva.shape, Xva.dtype, "y", yva.shape, yva.dtype)
    print("Test  group:", gte, "X", Xte.shape, Xte.dtype, "y", yte.shape, yte.dtype)
    print()

    print("=== SHAPE CHECKS ===")
    assert_3d(Xtr, "X_train")
    assert_3d(Xva, "X_val")
    assert_3d(Xte, "X_test")

    C_tr, T_tr = Xtr.shape[1], Xtr.shape[2]
    C_va, T_va = Xva.shape[1], Xva.shape[2]
    C_te, T_te = Xte.shape[1], Xte.shape[2]
    if (C_tr, T_tr) != (C_va, T_va) or (C_tr, T_tr) != (C_te, T_te):
        raise AssertionError(
            f"Inconsistent (C,T): train {(C_tr,T_tr)}, val {(C_va,T_va)}, test {(C_te,T_te)}"
        )
    print("✅ Shapes OK. (C,T) =", (C_tr, T_tr))
    print()

    print("=== LABEL CHECKS ===")
    cls_tr = label_report(ytr, "train")
    cls_va = label_report(yva, "val")
    cls_te = label_report(yte, "test")

    all_classes = np.unique(np.concatenate([cls_tr, cls_va, cls_te]))
    expected = np.arange(all_classes.min(), all_classes.max() + 1)
    if not np.array_equal(all_classes, expected):
        raise AssertionError(
            f"Label IDs not contiguous: got {all_classes.tolist()} expected {expected.tolist()}"
        )
    if all_classes.min() != 0:
        raise AssertionError("Expected labels to start at 0")

    print("✅ Labels look contiguous starting at 0. Classes:", all_classes.tolist())
    print()

    print("=== NUMERIC SANITY ===")
    numeric_sanity(Xtr, "X_train")
    numeric_sanity(Xva, "X_val")
    numeric_sanity(Xte, "X_test")
    print("✅ No NaN/Inf detected.")
    print()

    print("=== SAMPLE PREVIEW ===")
    idx = 0
    print("Sample 0 label:", int(ytr[idx]))
    print("Sample 0 X shape:", Xtr[idx].shape, "(expected (C,T))")
    print("First channel, first 10 timesteps:", np.round(Xtr[idx, 0, :10], 4))
    print()

    if os.path.isfile(meta_json):
        print("=== META.JSON ===")
        with open(meta_json, "r", encoding="utf-8") as f:
            meta = json.load(f)
        print(json.dumps(meta, indent=2))
        print()
    else:
        print("meta.json not found (optional).")

    print("✅ All checks passed.")


if __name__ == "__main__":
    main()

: 

In [ ]:
import os, textwrap, pathlib, json
py_path = "/mnt/data/test_basicmotions_hdf5.py"
with open(py_path, "w", encoding="utf-8") as f:
    f.write(script)

py_path

In [ ]:
import numpy as np
x = np.random.rand(2,3)
x

: 